# Gemma 2B: Base Model vs RLHF Instruct Model

This notebook loads both versions of Gemma 2B and lets you compare their responses to the same prompts.

- **Base model** (`google/gemma-2b`): pre-trained on text, just predicts the next token
- **Instruct model** (`google/gemma-2b-it`): fine-tuned with RLHF to follow instructions

> **Before running:** Make sure you have accepted the Gemma license at https://huggingface.co/google/gemma-2b

## Step 1: Install dependencies


In [ ]:
!pip install -q transformers accelerate bitsandbytes

## Step 2: Log in to Hugging Face

Get your token from https://huggingface.co/settings/tokens. Use a Read token.

In [ ]:
from huggingface_hub import login

login(token="hf_enter_your_token_from_huggingface")

## Step 3: Load the Base Model

This is Gemma 2B **before** RLHF. It only knows how to complete text.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

print("Loading base model...")

base_tokenizer = AutoTokenizer.from_pretrained("google/gemma-2b")
base_model = AutoModelForCausalLM.from_pretrained(
    "google/gemma-2b",
    device_map="auto",
    torch_dtype=torch.float16,
)

print("Base model loaded.")

## Step 4: Load the Instruct Model

This is Gemma 2B **after** RLHF. It has been fine-tuned to follow instructions.

In [ ]:
print("Loading instruct model...")

instruct_tokenizer = AutoTokenizer.from_pretrained("google/gemma-2b-it")
instruct_model = AutoModelForCausalLM.from_pretrained(
    "google/gemma-2b-it",
    device_map="auto",
    torch_dtype=torch.float16,
)

print("Instruct model loaded.")

## Step 5: Helper functions

In [ ]:
def run_base_model(prompt, max_new_tokens=200):
    """Run the base model — it will try to complete your text."""
    inputs = base_tokenizer(prompt, return_tensors="pt").to(base_model.device)
    outputs = base_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.7,
        pad_token_id=base_tokenizer.eos_token_id
    )
    return base_tokenizer.decode(outputs[0], skip_special_tokens=True)


def run_instruct_model(prompt, max_new_tokens=200):
    """Run the instruct model — it will try to answer your question."""
    inputs = instruct_tokenizer(prompt, return_tensors="pt").to(instruct_model.device)
    outputs = instruct_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.7,
        pad_token_id=instruct_tokenizer.eos_token_id
    )
    return instruct_tokenizer.decode(outputs[0], skip_special_tokens=True)


def compare(prompt):
    """Run the same prompt through both models and print the results."""
    print(f"PROMPT: {prompt}")
    print("-" * 60)
    print("BASE MODEL (pre-RLHF):")
    print(run_base_model(prompt))
    print()
    print("INSTRUCT MODEL (post-RLHF):")
    print(run_instruct_model(prompt))
    print("=" * 60)

## Step 6: Compare the models

Run the same prompts through both. Notice how differently they respond.


In [ ]:
# The base model will likely continue this as a travel blog or list
# The instruct model will answer it like an assistant
compare("What are good things to do in London?")